## 1. 预处理

### 1.1. 配置训练数据集和验证数据集

1. 手动配置：

将所有的训练集数据 (.wav 格式音频切片) 放到 `data/train/audio`。

将所有的验证集数据 (.wav 格式音频切片) 放到 `data/val/audio`。

2. 程序随机选择：

运行`python draw.py`,程序将帮助你挑选验证集数据（可以调整 `draw.py` 中的参数修改抽取文件的数量等参数）。

3. 文件夹结构目录展示：

* 单人物目录结构：

```
data
├─ train
│    ├─ audio
│    │    ├─ aaa.wav
│    │    ├─ bbb.wav
│    │    └─ ....wav
├─ val
│    ├─ audio
│    │    ├─ eee.wav
│    │    ├─ fff.wav
│    │    └─ ....wav
```

* 多人物目录结构：

```
data
├─ train
│    ├─ audio
│    │    ├─ 1
│    │    │   ├─ aaa.wav
│    │    │   ├─ bbb.wav
│    │    │   └─ ....wav
│    │    ├─ 2
│    │    │   ├─ ccc.wav
│    │    │   ├─ ddd.wav
│    │    │   └─ ....wav
│    │    └─ ...
|
├─ val
|    ├─ audio
│    │    ├─ 1
│    │    │   ├─ eee.wav
│    │    │   ├─ fff.wav
│    │    │   └─ ....wav
│    │    ├─ 2
│    │    │   ├─ ggg.wav
│    │    │   ├─ hhh.wav
│    │    │   └─ ....wav
│    │    └─ ...
```

In [ ]:
!python draw.py

### 1.2. 执行预处理

```bash
python preprocess.py -c configs/reflow.yaml
```

1. 默认配置适用于 RTX-4060 显卡训练 44.1khz 高采样率合成器。

2. 请保持所有音频切片的采样率与 yaml 配置文件中的采样率一致！如果不一致，程序可以跑，但训练过程中的重新采样将非常缓慢。（可选：使用 Adobe Audition™ 的响度匹配功能可以一次性完成重采样修改声道和响度匹配。）

3. 训练数据集的音频切片总数建议为约 1000 个，另外长音频切成小段可以加快训练速度，但所有音频切片的时长不应少于 2 秒。如果音频切片太多，则需要较大的内存，配置文件中将 `cache_all_data` 选项设置为 false 可以解决此问题。

4. 验证集的音频切片总数建议为 10 个左右，不要放太多，不然验证过程会很慢。

5. 如果您的数据集质量不是很高，请在配置文件中将 'f0_extractor' 设为 'rmvpe'.

6. 配置文件中的 ‘n_spk’ 参数将控制是否训练多说话人模型。如果您要训练**多说话人**模型，为了对说话人进行编号，所有音频文件夹的名称必须是**不大于 ‘n_spk’ 的正整数**。

In [ ]:
!python preprocess.py -c configs/reflow.yaml

## 2. 训练

```bash
python train_reflow.py -c configs/reflow.yaml
```

1. 训练开始后，每 ‘interval_val’ 步临时保存一个权重，每 ‘interval_force_save’ 步永久保存一个权重，可根据情况修改这两个配置项。

2. 可以随时中止训练，然后运行相同的命令来从最新保存的权重开始继续训练。

3. 微调 (finetune)：在中止训练后，重新预处理新数据集或更改训练参数（batchsize、lr 等），然后运行相同的命令。

In [ ]:
!python train_reflow.py -c configs/reflow.yaml

## 4. 推理

```bash
python main_reflow.py -i <input.wav> -m <model_ckpt.pt> -o <output.wav> -k <keychange (semitones)> -id <speaker_id> -step <infer_step> -method <method> -ts <t_start>
```

'infer_step' 为 rectified-flow ODE 的采样步数，'method' 为 'euler' 或 'rk4'，'t_start' 为 ODE 的起始时间点，需要大于或等于配置文件中的 `t_start`，建议保持相等（默认为 0.0）。

In [ ]:
!python main_reflow.py -i /root/sft/DDSP-SVC/input/情非得已_vocals_noreverb.wav -m /root/sft/DDSP-SVC/exp/reflow-test/20250903_102932/model_8000.pt -o /root/sft/DDSP-SVC/input/ai/ai_syz_情非得已.wav 